In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

supplier_snapshot = spark.table(
    "agentdb.gold.snapshot_supplier_risk"
)
delay_frequency = spark.table(
    "agentdb.features.feature_supplier_delay_frequency"
)

In [0]:
supplier_risk = (

    supplier_snapshot.alias("s")

    .join(
        delay_frequency.alias("d"),
        "supplier_key",
        "left"
    )

    .withColumn(

        "reliability_score",

        (
            lit(1.0)

            -
            coalesce(
                col(
                    "delay_frequency"
                ),
                lit(0.0)
            )
        )
    )

    .withColumn(

        "supplier_risk_score",

        (

            col(
                "disruption_probability"
            ) * 0.5

            +

            coalesce(
                col(
                    "delay_frequency"
                ),
                lit(0.0)
            ) * 0.3

            +

            col(
                "lead_time_risk"
            ) * 0.2
        )
    )

    .withColumn(

        "risk_level",

        when(
            col(
                "supplier_risk_score"
            ) > 0.8,
            "CRITICAL"
        )

        .when(
            col(
                "supplier_risk_score"
            ) > 0.6,
            "HIGH"
        )

        .when(
            col(
                "supplier_risk_score"
            ) > 0.4,
            "MEDIUM"
        )

        .otherwise(
            "LOW"
        )
    )

    .drop("created_at")
    
    .withColumn(
        "created_at",
        current_timestamp()
    )
)

In [0]:
(
    supplier_risk
    .withColumn(
        "lead_time_days",
        lit(None).cast("integer")
    )
    .select(
        "supplier_key",
        "lead_time_days",
        "disruption_probability",
        "delay_frequency",
        "reliability_score",
        "supplier_risk_score",
        "risk_level",
        "created_at"
    )
    .write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.intelligence.supplier_risk"
    )
)

In [0]:
display(
    supplier_risk
    .limit(10)
)